# External Macro Feature Analysis

This notebook validates that external macroeconomic features were merged correctly into the training data and explores whether those features help contextualize default risk over time.

## Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from config import CONFIG
from src.load_data import DataLoader

## Load Processed Training Data

Use the project DataLoader to load and filter the LendingClub dataset with the same logic used in training.

In [2]:
loader = DataLoader(CONFIG)
df = loader.load_and_filter_data()

print(f"Data shape: {df.shape}")
df.head()

Loading data according to paper methodology...
Resolved directory to CSV: /Users/priscillaashleywijaya/Desktop/General/NUS_Fintech_Society/CreditShield/credit-risk-prediction-project/training/data/accepted_2007_to_2018Q4.csv/accepted_2007_to_2018Q4.csv
Loading data with memory optimization...
Loading 68 of 68 essential columns
Error with optimized loading: cannot safely convert passed user dtype of int8 for float64 dtyped data in column 25
Falling back to standard loading...
✓ Loaded 2,260,701 rows, 68 columns
⚠️ External macro files not found: credit-risk-prediction-project/training/data/external_macro/FEDFUNDS.csv and/or credit-risk-prediction-project/training/data/external_macro/UNRATE.csv. Skipping external features.
✓ Filtered to years [2013, 2014]: 370,443 rows
✓ Removed incomplete loans: 358,244 remaining
Date range: 2013-01-01 to 2014-12-01
Data shape: (358244, 73)


,loan_amnt,term,int_rate,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,...,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,issue_date,issue_year,mths_since_last_delinq_missing,mths_since_last_major_derog_missing,mths_since_last_record_missing
1117060,10400.0,36 months,6.99,Truck Driver Delivery Personel,8 years,MORTGAGE,10.968216,Not Verified,Dec-2014,Charged Off,...,0.0,179407.0,15030.0,13000.0,11325.0,2014-12-01,2014.0,0,0,1
1117061,15000.0,60 months,12.39,MANAGEMENT,10+ years,RENT,11.264477,Source Verified,Dec-2014,Fully Paid,...,0.0,196500.0,149140.0,10000.0,12000.0,2014-12-01,2014.0,1,1,1
1117062,9600.0,36 months,13.66,Admin Specialist,10+ years,RENT,11.141876,Source Verified,Dec-2014,Fully Paid,...,0.0,52490.0,38566.0,21100.0,24890.0,2014-12-01,2014.0,1,1,1
1117063,7650.0,36 months,13.66,Technical Specialist,< 1 year,RENT,10.819798,Source Verified,Dec-2014,Charged Off,...,0.0,82331.0,64426.0,4900.0,64031.0,2014-12-01,2014.0,1,1,1
1117065,21425.0,60 months,15.59,Programming Analysis Supervisor,6 years,RENT,11.063524,Source Verified,Dec-2014,Fully Paid,...,0.0,57073.0,42315.0,15000.0,35573.0,2014-12-01,2014.0,0,0,1


## Check Macro Column Availability

Verify expected merged macro columns are present before continuing analysis.

In [3]:
expected_macro_cols = [
    "issue_date",
    "issue_month",
    "fed_funds_rate",
    "unemployment_rate",
    "fed_funds_rate_3m_change",
    "unemployment_rate_3m_change",
    "rate_tightening_flag",
    "unemployment_rising_flag",
]

present_expected = [c for c in expected_macro_cols if c in df.columns]
missing_expected = [c for c in expected_macro_cols if c not in df.columns]

print("Present expected columns:", present_expected)
print("Missing expected columns:", missing_expected)

if present_expected:
    df[present_expected].head(10)
else:
    raise ValueError("None of the expected macro columns were found in the loaded dataframe.")

Present expected columns: ['issue_date']
Missing expected columns: ['issue_month', 'fed_funds_rate', 'unemployment_rate', 'fed_funds_rate_3m_change', 'unemployment_rate_3m_change', 'rate_tightening_flag', 'unemployment_rising_flag']


## Missingness Check

Check missing proportions for core macro series and short-term change features.

In [4]:
missingness_cols = [
    "fed_funds_rate",
    "unemployment_rate",
    "fed_funds_rate_3m_change",
    "unemployment_rate_3m_change",
]

available_missingness_cols = [c for c in missingness_cols if c in df.columns]
if not available_missingness_cols:
    raise ValueError("No macro columns available for missingness checks.")

missing_props = df[available_missingness_cols].isna().mean().sort_values()
print("Missing proportions (ascending):")
print(missing_props)

ValueError: No macro columns available for missingness checks.

## Build Monthly Macro Table

Create a month-level macro table to support time-series and correlation analysis.

In [ ]:
required_for_monthly_macro = ["issue_month", "fed_funds_rate", "unemployment_rate"]
missing_for_monthly_macro = [c for c in required_for_monthly_macro if c not in df.columns]
if missing_for_monthly_macro:
    raise ValueError(f"Missing required columns for monthly macro table: {missing_for_monthly_macro}")

if not pd.api.types.is_datetime64_any_dtype(df["issue_month"]):
    df["issue_month"] = pd.to_datetime(df["issue_month"], errors="coerce")

monthly_macro = (
    df.groupby("issue_month", as_index=False)[["fed_funds_rate", "unemployment_rate"]]
    .first()
    .sort_values("issue_month")
    .reset_index(drop=True)
)

monthly_macro.head()

## Plot Macro Series Over Time

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(monthly_macro["issue_month"], monthly_macro["fed_funds_rate"], marker="o")
plt.title("Fed Funds Rate Over Issue Month")
plt.xlabel("Issue Month")
plt.ylabel("Fed Funds Rate")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(monthly_macro["issue_month"], monthly_macro["unemployment_rate"], marker="o")
plt.title("Unemployment Rate Over Issue Month")
plt.xlabel("Issue Month")
plt.ylabel("Unemployment Rate")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Detect Target Column

Prefer `is_default` when available; otherwise infer the most plausible binary default target and report the selected column.

In [ ]:
def is_binary_series(s: pd.Series) -> bool:
    non_null = s.dropna()
    if non_null.empty:
        return False
    unique_vals = set(non_null.unique())
    return len(unique_vals) <= 2 and unique_vals.issubset({0, 1, True, False})

candidate_priority = [
    "is_default",
    "target",
    "default_flag",
    "loan_default",
    "bad_loan",
    "default",
]

selected_target = None

if "is_default" in df.columns and is_binary_series(df["is_default"]):
    selected_target = "is_default"
else:
    for c in candidate_priority:
        if c in df.columns and is_binary_series(df[c]):
            selected_target = c
            break

if selected_target is None:
    binary_cols = [c for c in df.columns if is_binary_series(df[c])]

    if binary_cols:
        keyword_cols = [
            c for c in binary_cols
            if any(k in c.lower() for k in ["default", "target", "bad", "loan"])
        ]

        if keyword_cols:
            selected_target = sorted(keyword_cols)[0]
        else:
            scores = {}
            for c in binary_cols:
                non_null = df[c].dropna()
                positive_rate = float((non_null.astype(int) == 1).mean()) if len(non_null) > 0 else 0.0
                scores[c] = (len(non_null), abs(positive_rate - 0.5))
            selected_target = sorted(scores.keys(), key=lambda x: (-scores[x][0], scores[x][1]))[0]

if selected_target is None:
    raise ValueError(
        "Unable to detect a binary default target column. Checked standard names and binary columns; none matched."
    )

print(f"Selected target column: {selected_target}")

## Monthly Default Rate Over Time

In [ ]:
if "issue_month" not in df.columns:
    if "issue_date" in df.columns:
        df["issue_month"] = pd.to_datetime(df["issue_date"], errors="coerce").dt.to_period("M").dt.to_timestamp()
    else:
        raise ValueError("Neither 'issue_month' nor 'issue_date' exists; cannot compute monthly default rate.")

monthly_default = (
    df.groupby("issue_month", as_index=False)[selected_target]
    .mean()
    .rename(columns={selected_target: "monthly_default_rate"})
    .sort_values("issue_month")
    .reset_index(drop=True)
)

monthly_default.head()

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(monthly_default["issue_month"], monthly_default["monthly_default_rate"], marker="o")
plt.title("Monthly Default Rate Over Issue Month")
plt.xlabel("Issue Month")
plt.ylabel("Default Rate")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Merge Monthly Macro and Monthly Default

Join month-level macro features with month-level default rates for direct comparison.

In [ ]:
monthly_merged = monthly_macro.merge(monthly_default, on="issue_month", how="inner")
monthly_merged.head()

## Correlation Analysis

Correlation values show linear association direction and strength between monthly macro levels and monthly default rate.

In [ ]:
corr_cols = ["fed_funds_rate", "unemployment_rate", "monthly_default_rate"]
missing_corr_cols = [c for c in corr_cols if c not in monthly_merged.columns]
if missing_corr_cols:
    raise ValueError(f"Missing columns for correlation analysis: {missing_corr_cols}")

corr_matrix = monthly_merged[corr_cols].corr()
print(corr_matrix)

## Bucket Analysis

Compare average default rates across quartiles of macro conditions.

In [ ]:
if "unemployment_rate" not in df.columns or "fed_funds_rate" not in df.columns:
    raise ValueError("Both 'unemployment_rate' and 'fed_funds_rate' are required for bucket analysis.")

bucket_df = df[["unemployment_rate", "fed_funds_rate", selected_target]].copy()
bucket_df = bucket_df.dropna(subset=["unemployment_rate", "fed_funds_rate", selected_target])

if bucket_df.empty:
    raise ValueError("No non-missing rows available for bucket analysis.")

bucket_df["unemployment_rate_bucket"] = pd.qcut(
    bucket_df["unemployment_rate"], q=4, duplicates="drop"
)
unemployment_bucket_default = (
    bucket_df.groupby("unemployment_rate_bucket", observed=False)[selected_target]
    .mean()
    .rename("default_rate")
    .reset_index()
)

bucket_df["fed_funds_rate_bucket"] = pd.qcut(
    bucket_df["fed_funds_rate"], q=4, duplicates="drop"
)
fed_bucket_default = (
    bucket_df.groupby("fed_funds_rate_bucket", observed=False)[selected_target]
    .mean()
    .rename("default_rate")
    .reset_index()
)

print("Default rate by unemployment quartile:")
print(unemployment_bucket_default)
print()
print("Default rate by fed funds quartile:")
print(fed_bucket_default)

## Flag Analysis

Evaluate default rates when macro trend flags indicate tightening or rising unemployment.

In [ ]:
flag_cols = ["unemployment_rising_flag", "rate_tightening_flag"]
missing_flag_cols = [c for c in flag_cols if c not in df.columns]
if missing_flag_cols:
    raise ValueError(f"Missing required flag columns: {missing_flag_cols}")

unemp_flag_default = (
    df.groupby("unemployment_rising_flag", dropna=False)[selected_target]
    .mean()
    .rename("default_rate")
    .reset_index()
)

tightening_flag_default = (
    df.groupby("rate_tightening_flag", dropna=False)[selected_target]
    .mean()
    .rename("default_rate")
    .reset_index()
)

print("Default rate by unemployment_rising_flag:")
print(unemp_flag_default)
print()
print("Default rate by rate_tightening_flag:")
print(tightening_flag_default)

## Final Summary

In [ ]:
summary = {
    "macro_missingness": missing_props.to_dict(),
    "fed_funds_min": float(df["fed_funds_rate"].min()) if "fed_funds_rate" in df.columns else None,
    "fed_funds_max": float(df["fed_funds_rate"].max()) if "fed_funds_rate" in df.columns else None,
    "unemployment_min": float(df["unemployment_rate"].min()) if "unemployment_rate" in df.columns else None,
    "unemployment_max": float(df["unemployment_rate"].max()) if "unemployment_rate" in df.columns else None,
    "n_unique_issue_months": int(df["issue_month"].nunique()) if "issue_month" in df.columns else None,
}

print(summary)

## Conclusion

The merge appears successful if the expected macro columns are present and missingness in core macro variables is low. The time-series, bucket, and flag analyses provide a quick check for visible variation in default rates across different macro conditions.